In [ ]:
# Install yt-dlp to download the video/audio and Whisper for ASR
!pip install -q yt-dlp openai-whisper

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 182.3/182.3 kB 7.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 27.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 40.0 MB/s eta 0:00:00


In [ ]:
%%capture
!pip install yt-dlp transformers torch accelerate
!apt-get update && apt-get install -y ffmpeg

import os
import torch
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline
import yt_dlp

In [ ]:
def download_youtube_audio(video_url, output_dir="transcription_data"):
    """Downloads the audio track of a YouTube video and saves it as an MP3."""
    os.makedirs(output_dir, exist_ok=True)

    ydl_opts = {
        'format': 'bestaudio/best',
        'postprocessors': [{
            'key': 'FFmpegExtractAudio',
            'preferredcodec': 'mp3',
            'preferredquality': '192',
        }],
        # Saves file as video_id.mp3 inside the output directory
        'outtmpl': os.path.join(output_dir, '%(id)s.%(ext)s'),
        'quiet': True
    }

    print(f"Downloading audio from: {video_url}")
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        info = ydl.extract_info(video_url, download=True)
        video_id = info['id']
        expected_filename = os.path.join(output_dir, f"{video_id}.mp3")
        return expected_filename, info.get('title', 'Unknown Title')

In [ ]:
# Hardware selection
device = "cuda:0" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

print(f"Loading Whisper model on: {device}")
model_id = "openai/whisper-large-v3"

model = AutoModelForSpeechSeq2Seq.from_pretrained(
    model_id, torch_dtype=torch_dtype, low_cpu_mem_usage=True, use_safetensors=True
).to(device)

processor = AutoProcessor.from_pretrained(model_id)

# The pipeline handles chunking automatically if chunk_length_s is set
asr_pipeline = pipeline(
    "automatic-speech-recognition",
    model=model,
    tokenizer=processor.tokenizer,
    feature_extractor=processor.feature_extractor,
    chunk_length_s=30,      # Automatically cuts audio into 30s chunks
    batch_size=8,           # Processes chunks in parallel for speed
    torch_dtype=torch_dtype,
    device=device,
)
print("Model loaded successfully!")

Loading Whisper model on: cuda:0


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/1.27k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/1259 [00:00<?, ?it/s]

...

merges.txt:   0%|          | 0.00/494k [00:00<?, ?B/s]

normalizer.json:   0%|          | 0.00/52.7k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/34.6k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.07k [00:00<?, ?B/s]

Model loaded successfully!


In [ ]:
# 1. Put your 3 YouTube URLs here
video_urls = [
    "https://www.youtube.com/watch?v=-URJ4t1q6ic",
    "https://www.youtube.com/watch?v=1AO3UDuwEBQ",
    "https://www.youtube.com/watch?v=1j8fCkoI2J0"
]

# 2. Run execution loop
for url in video_urls:
    try:
        # Download
        audio_path, video_title = download_youtube_audio(url)
        print(f"\nProcessing Transcription for: '{video_title}'...")

        # Transcribe (uncomment/change language argument if needed)
        result = asr_pipeline(
            audio_path,
            # generate_kwargs={"language": "hi"} # Explicitly force language if needed
        )

        # 3. Print out your text or save it to a file
        print(f"--- Transcript for '{video_title}' ---")
        print(result["text"])
        print("-" * 50)

        # Optional: Save transcript to a local text file
        txt_filename = f"{audio_path.rsplit('.', 1)[0]}_transcript.txt"
        with open(txt_filename, "w", encoding="utf-8") as f:
            f.write(result["text"])

    except Exception as e:
        print(f"Failed to process {url}. Error: {e}")


Processing Transcription for: 'Krishi Darshan कद्दू वर्गीय सब्जियों की बीजाई'...
Failed to process https://www.youtube.com/watch?v=-URJ4t1q6ic. Error: CUDA out of memory. Tried to allocate 148.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 123.81 MiB is free. Including non-PyTorch memory, this process has 14.44 GiB memory in use. Of the allocated memory 13.49 GiB is allocated by PyTorch, and 834.08 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)



Processing Transcription for: 'Krishi Darshan/First Aid after Accident During use of Agricultural Machinery'...
Failed to process https://www.youtube.com/watch?v=1AO3UDuwEBQ. Error: CUDA out of memory. Tried to allocate 148.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 123.81 MiB is free. Including non-PyTorch memory, this process has 14.44 GiB memory in use. Of the allocated memory 13.53 GiB is allocated by PyTorch, and 794.53 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)



Processing Transcription for: 'Krishi Darshan -  Mango & Lemon Cultivation'...
Failed to process https://www.youtube.com/watch?v=1j8fCkoI2J0. Error: CUDA out of memory. Tried to allocate 148.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 123.81 MiB is free. Including non-PyTorch memory, this process has 14.44 GiB memory in use. Of the allocated memory 13.53 GiB is allocated by PyTorch, and 794.53 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)
